# Ring yields 1 — species partition functions

Step 1 of the equilibrium ring-yield pipeline (`docs/ring_yield_plan.md`).

1. Fill the (θ, E_AB) scan of ln q_s for monomer, open chains and rings (`simulation/runners/scan_partition.py`). Each point is cached as `data/partition/oa<θ>_EAB<E>.npz`; rerunning only computes missing points. For the full grid use SLURM (`sbatch simulation/runners/slurm/submit_partition.sh`) and come back here.
2. Check the chain-growth estimates against independent Cartesian sampling (`analysis/direct_z.py`) for n = 3–5 (cached).
3. Look at the mode picture: harmonic vs sampled ln q_n, the soft-mode integral, and the per-bond residual b_n.

q_s do not depend on density, so the density sweeps of notebook 2 need only this scan.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'base').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from base.jax_setup import configure_jax
configure_jax()
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
from base.paths import PARTITION_DIR, PLOTS_DIR, DATA_DIR, ensure_dir
FIG_DIR = ensure_dir(PLOTS_DIR / 'ring_yields')

## 1. Scan (cached)
`RUN_LOCAL=False` only reads what is on disk. The default grid is `base.grids.PARTITION_*`; each point takes ~2 min on one core.

In [ ]:
from base.grids import PARTITION_ANGLES_DEG, PARTITION_EAB_VALUES
from simulation.runners.scan_partition import run_grid, point_path

RUN_LOCAL = False          # True: compute missing points here
WORKERS = 8
if RUN_LOCAL:
    run_grid(workers=WORKERS)
done = [(a, e) for a in PARTITION_ANGLES_DEG for e in PARTITION_EAB_VALUES if point_path(a, e).exists()]
print(f'{len(done)} / {len(PARTITION_ANGLES_DEG) * len(PARTITION_EAB_VALUES)} points on disk')

In [ ]:
from analysis.yield_maps import load_scan, plot_lnq_per_particle
scan = load_scan()
print('theta:', scan.thetas_deg, '\nE_AB:', scan.eabs, '\nspecies:', len(scan.labels))

### Sampling quality
Kish ESS of every ring at every point. Strained rings far from their matching angle sample poorly, but their yields are negligible; notebook 2 tracks how much yield sits in species below the cut.

In [ ]:
rings = [l for l in scan.labels if l.startswith('ring_')]
fig, axes = plt.subplots(2, (len(rings) + 1) // 2, figsize=(3.2 * ((len(rings) + 1) // 2), 5.5),
                         sharex=True, sharey=True, constrained_layout=True)
for ax, l in zip(axes.flat, rings):
    s = list(scan.labels).index(l)
    m = ax.pcolormesh(scan.thetas_deg, scan.eabs, np.log10(np.maximum(scan.ess[..., s], 1)),
                      vmin=0, vmax=4, cmap='Blues', shading='nearest')
    ax.set_title(l, fontsize=9); ax.set_yscale('log')
fig.colorbar(m, ax=axes, label='log10 ESS')
fig.supxlabel('θ (deg)'); fig.supylabel('E_AB')
plt.show()

### ln q_n / n across θ
Per-particle free energy of each ring; rings peak near their matching angle θ_n.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, E in zip(axes, [5.0, 10.0, 20.0]):
    plot_lnq_per_particle(scan, E, ax=ax)
fig.tight_layout(); fig.savefig(FIG_DIR / 'lnq_per_particle.png', dpi=150); plt.show()

## 2. Chain growth vs direct Cartesian sampling (cached)

In [ ]:
from analysis.partition import Model, estimate_ring
from analysis.direct_z import direct_ln_q

VAL_PATH = DATA_DIR / 'partition_validation' / 'direct_vs_smc.npz'
CASES = [(60, 10, 3), (60, 25, 3), (90, 10, 4), (90, 25, 4), (80, 10, 4), (108, 10, 5), (100, 15, 5)]
if VAL_PATH.exists():
    val = dict(np.load(VAL_PATH))
else:
    rows = []
    for th, E, n in CASES:
        m = Model(np.deg2rad(th), float(E))
        s = estimate_ring(m, n, n_walkers=16000, rng=1)
        d = direct_ln_q(m, n, n_samples=200_000)
        rows.append((th, E, n, s.lnq, s.lnq_err, d[0], d[1]))
    val = dict(zip(['theta', 'E_AB', 'n', 'smc', 'smc_err', 'direct', 'direct_err'], np.array(rows).T))
    ensure_dir(VAL_PATH.parent); np.savez(VAL_PATH, **val)
for r in zip(*[val[k] for k in ['theta', 'E_AB', 'n', 'smc', 'smc_err', 'direct', 'direct_err']]):
    print('θ=%3.0f E=%4.1f n=%d  smc %.3f±%.3f  direct %.3f±%.3f  diff %+.3f' % (*r, r[3] - r[5]))

## 3. Mode picture
`Z_n = e^{−βE0} A (J̃/σ_n) Π_stiff √(2π/βλ) I_soft e^{n b_n}`. Compare the full harmonic estimate with the sampled ln q_n, and the soft-mode integral with its Gaussian approximation.

In [ ]:
def mode_grid(key):
    out = np.full(scan.shape + (len(scan.modes[next(iter(scan.modes))]['n']),), np.nan)
    for (th, E), md_ in scan.modes.items():
        i, j = np.searchsorted(scan.eabs, E), np.searchsorted(scan.thetas_deg, th)
        out[i, j] = np.where(md_['valid'], md_[key], np.nan)
    return out

mode_n = scan.modes[next(iter(scan.modes))]['n']
lnq_ring = np.stack([scan.lnq[..., list(scan.labels).index(f'ring_{n}')] for n in mode_n], -1)
harm, b_n = mode_grid('ln_q_harmonic'), mode_grid('b_n')
soft, softg = mode_grid('ln_I_soft'), mode_grid('ln_soft_gaussian')

i10 = int(np.argmin(np.abs(scan.eabs - 10)))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
from analysis.yield_maps import ring_color
for k, n in enumerate(mode_n[:6]):
    c = ring_color(int(n))
    axes[0].plot(scan.thetas_deg, lnq_ring[i10, :, k] - harm[i10, :, k], color=c, lw=2, label=f'n={n}')
    axes[1].plot(scan.thetas_deg, soft[i10, :, k] - softg[i10, :, k], color=c, lw=2, label=f'n={n}')
    axes[2].plot(scan.thetas_deg, b_n[i10, :, k], color=c, lw=2, label=f'n={n}')
axes[0].set_ylabel('ln q (sampled) − ln q (harmonic)')
axes[1].set_ylabel('ln I_soft − Gaussian soft modes')
axes[2].set_ylabel('b_n (per bond)')
for ax in axes:
    ax.set_xlabel('θ (deg)'); ax.grid(alpha=0.25, lw=0.5); ax.axhline(0, color='k', lw=0.5)
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle(f'E_AB = {scan.eabs[i10]:g}'); fig.tight_layout()
fig.savefig(FIG_DIR / 'mode_breakdown.png', dpi=150); plt.show()